# Lesson 4: Gridded Data with Xarray

In this lesson, we are going to start looking at gridded data. This is data that can be stored in a multidimenaional array: think latitude, longitude, and time, although it doesn't have to be these three dimensions (or even just three dimensions!). 

In these lessons, when working with general multi-dimensional (gridded) data we'll be using `xarray`, which is a great library with seemingly endless possibilities for processing and analyzing gridded data. The learning curve can be a little steep but stick with it you will learn to love `xarray`.


### Learning outcomes

After this lesson, you should be able to:

1) Evaluate the contents of a dataset using its metadata attributes.
2) Use xarray to read gridded data in a range of common file formats. 
3) Subset multi-dimensional data to a geographic region.
4) Mask values in multi-dimensional data based on varied conditions.
5) Create one- and two-dimensional plots of an xarray DataArray.

:::{danger}
## Entry ticket!

Before we start, discuss with your neighbor for 1 minute the coding concept (or Python library) you think is most relevant to your SARP project.
:::

## Introduction to Xarray

`xarray` is a great (the?) library for working with gridded datasets. It has many built in analysis methods, nice visualization defaults, and it was built by the scientific community. `xarray` is built on top of another library called `numpy`. `xarray` takes the `numpy` arrays and makes them easier to work with by adding labels to the axes. This is a small change but it has a huge effect on the ease of working with data.

`xarray` allows us to work with labelled, multi-dimensional data. What does that mean? There are two parts:
1. **Multi-dimensional:** the data is organized into a grid and can have any number of dimensions. For example, data might be organized by latitude, longitude, and altitude (3 dimensions). This is often visually represented as a cube of data.
2. **Labelled:** the cube of data isn't just any old cube - each dimension has associated values. For example, we know the values of latitude, longitude, and altitude that the data is describing.

## Anatomy of a DataArray

The first core `xarray` data structure is a `DataArray`. Typically, you will be reading in existing data sources into `DataArrays` in Python. But let's start by briefly creating a `DataArray` from scratch with some fake data for air temperature. To construct a `DataArray` from scratch, we need:
* `data`: values for the variables of interest (in this case temperature), arranged in the grid defined by our dimensions and coordinates
* `dims`: the dimensions by which our data is arranged (in this case latitude and longitude)
* `coords`: the coordinate values of our dimensions (in this case, exactly which latitudes and longitudes our data are showing)

In [ ]:
import numpy as np
import xarray as xr

# Data values
air_temps = np.array([
    [80, 80, 81, 83, 82, 80], [81, 80, 81, 83, 83, 82], [81, 81, 83, 84, 84, 83], 
    [79, 80, 83, 85, 85, 85], [78, 77, 80, 85, 86, 87]
])

# Coordinate values
lats = [36, 37, 38, 39, 40]
lons = [-80, -79, -78, -77, -76, -75]

# Assemble the DataArray
temps = xr.DataArray(data=air_temps, 
                     dims=['latitude', 'longitude'], 
                     coords=[lats, lons])


temps

In [ ]:
# Easy quick plotting with xarray
temps.plot()

In [ ]:
# Data array attributes
print('DIMS', temps.dims)
print('COORDS', temps.coords)
print('DATA', temps.data)

In [ ]:
# Numpy ndarray of the values 
temps.values

:::{admonition}
### Exercise: Inspecting a DataArray

The following `DataArray` contains values for wind speeds. 
1. What are the dimensions?
2. What are the coordinates?
3. What is the highest and lowest wind speed in the array?
4. Finally, make a hand-drawn sketch of the DataArray.

![example output](./images/example_2d_repr.png)

## Real-world gridded data: MUR sea-surface temperatures

Typically, we won't be creating our own DataArray from scratch. Instead, we'll be opening existing multi-dimensional datasets. Let's open up the [GHRSST Level 4 MUR](https://podaac.jpl.nasa.gov/dataset/MUR25-JPL-L4-GLOB-v04.2), a global sea surface temperature (SST) dataset from NASA JPL.

In [ ]:
sst = xr.open_dataset('./data/20250503090000-JPL-L4_GHRSST-SSTfnd-MUR25-GLOB-v02.0-fv04.2.nc')
sst

## Metadata

A new thing you will notice about this real dataset is that it has Attributes. Attributes are information about the data that isn't actually the data itself, what we commonly call **metadata**.

:::{hint} Vocabulary
**Metadata:** information that describes your data, but isn’t the actual data values.
:::

:::{admonition} 
### Exercise

Before we work with our data, it is always good to start by checking the metadata. Using the SST data we just opened, can you find:
- How many dimensions does it have and what is the shape?
- How many variables are there? What are they?
- What geographic area is covered by this dataset?
- On what date(s) was the data captured?
- What are the units of the `'analysed_sst'` data? (Hint: you will need to first convert the DataSet into a DataArray by selecting the `'analysed_sst'` variable.)
- What is the source of the `sea_ice_fraction'` data? (Hint: you will need to first convert the DataSet into a DataArray by selecting the `'sea_ice_fraction'` variable.)
:::

## Inspecting the data values

Now that we've looked at the metadata and got a grasp of the data we are looking at, we can move ahead and look at the values contained in the data. One of the ways to inspect our data right after opening is to plot it.

In [ ]:
sst.plot()

:::{warning}
## Bug alert!

Oops! What happened?

`ValueError: Dataset.plot cannot be called directly. Use an explicit plot method, e.g. ds.plot.scatter(...)`

Here we see that xarray doesn’t like that we have given the `.plot()` method for a Dataset. What if we gave it a DataArray instead? Let’s extract the `'analysed_sst'` variable before we attempt to plot.

:::

In [ ]:
# Note how this gives us a data array from our original dataset
sst_da = sst['analysed_sst']
sst_da

In [ ]:
# Since the SST are in Kelvin, let's convert to F - we can simply apply math formula across all values
sst_celsius = sst_da - 273.15

# We will also update the metadata attributes so it is consistent with the values
sst_celsius.attrs['units'] = 'C'
sst_celsius

In [ ]:
# Lets try to plot again, having selected the sst variable in the dataset
# What do you notice about global land areas? 
sst_celsius.plot()

## Indexing and selecting values from a DataArray

One of the great things about having labelled data is that we can use the labels to extract a smaller amount of data from a larger dataset. This process of extracting some data from a larger amount of data is called subsetting.

Often we want to find not just a single value, but data over a larger area. For example, maybe the data file that you downloaded has data for the whole world but you only want data over the state of California. To achieve that we use a `slice()`.

:::{hint} Vocabulary
**Subsetting:** taking a large dataset and picking out just the bit of data that you need for your analysis. We do this in `xarray` using the `.sel()` (selecting by label) and `.isel()` (selecting by index) syntax, often combined with `slice()`.

In [ ]:
# Shape of the global data
sst_celsius.shape

In [ ]:
# Using sel and slice to subset by lat/lon coordinates
sst_celsius.sel(lat=slice(5, 65), lon=slice(-85, -36))

In [ ]:
# Plot the subset area
sst_celsius.sel(lat=slice(5, 65), lon=slice(-85, -36)).plot()

In [ ]:
# Now using isel - selecting by index position
# Note how we are now starting from the first lat and first lon value (using index 0)
# In our case, this is located at 90 degrees S, 180 degrees W
sst_celsius.isel(lat=slice(0, 100), lon=slice(0, 100)).plot()

## Xarray functions/methods

Xarray comes with a powerful suite of functions and methods that combine the labeled dimensions and metadata handling of NetCDF-style datasets with many of the familiar operations available in NumPy and pandas. 

#### Aggregations

Aggregations with xarray work in a similar way to the aggregations we have seen with pandas (the function names themselves are deliberately the same). Since we are working with multi-dimensional data, we now have to additionally specify the dimension(s) we are aggregating over. Commonly, we might want to aggregate over either time (`dim='time'`) or space (`dim=['lat', 'lon']`):

| Function/Method | Description | Example |
|----------|-------------|---------|
| `mean()` | Calculate mean along one or more dimensions | `data.mean(dim='time')`, `data.mean(dim=['lat', 'lon'])` |
| `min()` | Calculate max along one or more dimensions | `data.min(dim='time')` |
| `max()` | Calculate max along one or more dimensions | `data.max(dim='time')` |
| `sum()` | Sum values along one or more dimensions | `data.sum(dim='time')` |
| `count()` | Count values along one or more dimensions | `data.count(dim='time')` |

### Other common functions/methods

Other common tasks on gridded data include filtering / masking certain values, resampling time series, changing spatial resolution, interpolating between coordinates, combining variables, and performing mathematical operations. Here are just some of the functions you might find most relevant:

| Function/Method | Description | Example |
|----------|-------------|---------|
| `squeeze()` | Drop empty dimensions | `data.squeeze()` |
| `where()` | Keep values meeting a condition, mask others | `data.where(ds.precip > 0)` |
| `resample()` | Aggregate or interpolate along time | `data.resample(time='M').mean()` |
| `coarsen()` | Aggregate to a lower spatial or temporal resolution | `data.coarsen(lat=2, lon=2).mean()` |
| `groupby()` | Group data by a coordinate or attribute | `data.groupby('time.month').mean()`, `data.groupby('time.season').mean()` |
| `rolling()` | Apply moving-window operations | `data.rolling(time=7).mean()` |
| `interp()` | Interpolate to new coordinates | `data.interp(lat=35.5, lon=-97.5)` |

In [ ]:
# Since our time dimension only has one time value, it doesn't really hold any useful dimension coordinate values
# We can drop this empty dimension using squeeze() 
sst_celsius = sst_celsius.squeeze()
sst_celsius

In [ ]:
# Let's try out where() to mask values that do not meet a condition of our choice
# Note how we have masked out areas with SST below 20 C
sst_celsius.where(sst_celsius > 20).plot()

In [ ]:
# Aggregations - maximum aggregated over the latitude dimension 
# I.e. max over all latitudes for each longitude coordinate value
# Note what dimensions and shape we get back
sst_celsius.max(dim='lat')

# Note that we can use axis argument to do the same thing, using the index of the dimension
# sst_celsius.max(axis=1)

In [ ]:
# Plot the aggregation maximum over latitude dimension
# Note how xarray DataArray.plot() responds differently depending on the number of dimensions we have
sst_celsius.max(dim='lat').plot()

:::{admonition} 

### Exercise: Working with DataArrays

Now it's your turn to work with our SST DataArray:

1. Subset the global SST data (`sst_celsius`) to a geographical region of your choice. Check the shape of your subsetted data.
2. Mask out areas of your subsetted data that have a SST below the median SST for your region. Create a plot of the masked data using a colormap of your choice: https://matplotlib.org/stable/users/explain/colors/colormaps.html.  
3. Using your masked subsetted data, extract a one-dimensional DataArray along a single longitude coordinate value of your choice. Create a (line) plot. 

:::

:::{dropdown} 

### Solution

```
# Subset area
sst_sub = sst_celsius.sel(lat=slice(5, 65), lon=slice(-85, -36))
print(sst_sub.shape)

# Mask out values lower than the median
sst_masked = sst_sub.where(sst_sub > sst_sub.median())

# Plot map
sst_masked.plot(cmap='plasma')

# Plot line showing SST along a single longitude (here the 100th index longitude)
sst_masked.isel(lon=100).plot()
```

:::{danger}
## Exit ticket!

Before you leave this lesson, please [submit your responses here](https://forms.gle/cVjyNwBZyUycm6RT6) to three quick questions: 

* How much of this lesson's content do you feel comfortable with? 0-10, with 10 being all of the content.
* How was the pace of this lesson for you? 1) Too slow; 2) About right; 3) Too fast.
* In a few words or less, what single concept was most challenging for you in this lesson?

:::

:::{hint}
## Further reading

* [Xarray tutorial](https://xarray-contrib.github.io/xarray-tutorial/)
* [UW Geohackweek Xarray tutorial](https://geohackweek.github.io/nDarrays/)
* SARP Programming Lessons [3.2 (band math)](https://nasa-sarp.github.io/sarp_lessons/sarp_lessons/1_python_progression/3-gridded_data/3-2_xarray_computation.html) and [3.3 (group by)](https://nasa-sarp.github.io/sarp_lessons/sarp_lessons/1_python_progression/3-gridded_data/3-3_xarray_groupby.html)

:::{hint}
## File formats: NetCDF, HDF5, GRIB, Zarr

We've just worked with a real world gridded dataset. Did you notice the file extension? Files such as `.nc`, `.h5`, `.grib`, and `.zarr` are all common formats for storing raster or multidimensional data.

These formats are widely used in Earth science because they can efficiently store data that vary across space, time, and sometimes additional dimensions (e.g., depth, wavelength, ensemble member).

| Format | Typical Extension | Common Uses |
|:----------|:----------|:----------|
| NetCDF | `.nc` | Climate, weather, oceanography, remote sensing |
| HDF5 | `.h5`, `.hdf5` | Satellite products, scientific data archives |
| GRIB | `.grib`, `.grb` | Numerical weather prediction data |
| Zarr | `.zarr` | Cloud-native and large multidimensional datasets |

### NetCDF and HDF5

NetCDF (Network Common Data Form) and HDF5 (Hierarchical Data Format) are among the most common scientific data formats. Both can store multiple variables, dimensions, metadata, and coordinate systems within a single file. For example, a NetCDF file might contain daily precipitation with dimensions `(time, latitude, longitude)` along with metadata describing units, projection information, and variable names.:

### GRIB

GRIB (GRIdded Binary) is commonly used for weather forecast and reanalysis products. Many operational weather forecast centers distribute forecast datasets in GRIB format. GRIB files are highly optimized for storing large weather datasets but can be more difficult to work with than NetCDF.

### Zarr

Zarr is a newer format designed for cloud storage and parallel computing. Instead of storing everything in a single file, a Zarr dataset is typically stored as a directory containing many small chunks of data. This structure makes it possible to efficiently access only the portions of a dataset that are needed, which is particularly useful for very large datasets.

#### Reading gridded data with xarray

One of the advantages of `xarray` is that it provides a consistent interface for many different file formats.

```python
import xarray as xr

# NetCDF
ds = xr.open_dataset("precipitation.nc")

# HDF5 (many HDF products can also be opened this way)
ds = xr.open_dataset("satellite_data.h5")

# GRIB (requires cfgrib package)
ds = xr.open_dataset("forecast.grib", engine="cfgrib")

# Zarr
ds = xr.open_zarr("dataset.zarr")
```

#### Writing gridded data with xarray

Like pandas provides `to_csv()`, xarray includes methods for saving datasets to common scientific formats:

```python
# Save to NetCDF
ds.to_netcdf("output.nc")

# Save to Zarr
ds.to_zarr("output.zarr")
```
:::